In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general",
)

In [5]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    #api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [8]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

In [9]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [10]:
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

response = agent.invoke({"messages": [multimodal_message]}, config)

In [13]:
# 友好打印
for message in response['messages']:
    message.pretty_print()


================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_c6f0c224720a408291bdfa2b)
 Call ID: call_c6f0c224720a408291bdfa2b
  Args:
    query: 三文鱼 鸡胸肉 西兰花 蘑菇 彩椒 健康食谱
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "三文鱼 鸡胸肉 西兰花 蘑菇 彩椒 健康食谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://cookpad.com/tw/%E9%A3%9F%E8%AD%9C/13394466", "title": "三文魚&雞胸肉食譜與作法by Suky   - Cookpad", "content": "已收藏    \n\n收藏食譜以便日後查看\n\n## 料理步驟\n\n25mins\n\n1. 1\n\n   將三文魚和雞胸肉清理乾淨 加入適量鹽和黑胡椒醃製30分鐘\n2. 2\n\n   西蘭花切一口的大小 番茄切片 薯仔切片（用水浸下）\n3. 3\n\n   焗爐盤刷上一點油 放上番茄土豆和西蘭花 將三文魚和雞胸肉放在蔬菜上

In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第1道菜，可以说的更详细点吗？")]},
    config
)

# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

# 🍽️ 香烤三文鱼鸡胸肉拼盘 - 详细烹饪指南

## 一、食材准备清单（2-3人份）

### 主料
| 食材 | 用量 | 处理说明 |
|------|------|----------|
| 三文鱼排 | 2块（约200-250g） | 去鳞、去刺，保留鱼皮 |
| 鸡胸肉 | 1块（约200g） | 去筋膜，切成厚片 |
| 西兰花 | 半颗（约150g） | 切成小朵 |
| 蘑菇 | 8-10个 | 对半切开或整颗 |
| 彩椒 | 红/黄各半个 | 去籽切块 |
| 小番茄 | 10-12个 | 洗净保留整颗 |
| 柠檬 | 半个 | 切片备用 |

### 调味料
| 调料 | 用量 |
|------|------|
| 橄榄油 | 2汤匙 |
| 海盐 | 1茶匙 |
| 黑胡椒碎 | ½茶匙 |
| 蒜末 | 2瓣（可选） |
| 迷迭香/百里香 | 2-3枝（可选） |
| 黄油 | 10g（可选，增加香气） |

---

## 二、详细烹饪步骤

### 📋 步骤1：准备工作（10分钟）
```
1. 将烤箱预热至 180°C（上下火）
2. 三文鱼用厨房纸吸干表面水分
3. 鸡胸肉用刀背轻轻拍打，使厚度均匀
4. 所有蔬菜洗净、切好备用
```

### 📋 步骤2：腌制肉类（15-30分钟）
```
1. 三文鱼两面均匀撒上盐和黑胡椒
2. 鸡胸肉同样调味，可加入少许蒜末
3. 每块肉上放1-2片柠檬
4. 淋上1汤匙橄榄油
5. 盖上保鲜膜，室温腌制15-30分钟
   （时间越长越入味，但不超过1小时）
```

### 📋 步骤3：处理蔬菜（5分钟）
```
1. 西兰花放入沸水中焯烫1分钟，捞出沥干
   （这样烤出来颜色更翠绿）
2. 蘑菇、彩椒、小番茄放入碗中
3. 加入1汤匙橄榄油、少许盐和黑胡椒
4. 轻轻拌匀
```

### 📋 步骤4：组装烤盘（3分钟）
```
1. 烤盘铺上烘焙纸或锡纸（方便清洗）
2. 先铺蔬菜在烤盘底部
   - 西兰花、蘑菇、彩椒均匀分布
   - 小番茄放在空隙处
3. 将腌好的三文鱼和鸡胸肉放在蔬菜上面
   - 三